# Background (sky) trials on the corners and centre of a detector (v15: light pass clean-up)

Fits five cutouts -- the four corners and the centre of one LSSTCam detector, each a real tile
(~372 px, 30 px padding) -- with the fast preset, once per trial, and shows for every trial:

* a table: sky subtracted, median residual/sigma **between stars** (model < 0.5 sigma) and **on
  stars** (model > 4 sigma), robust residual scatter, number of sources, psfMag - Gaia G;
* per cutout, one row per trial: data | model | residual | residual/sigma | residual/sigma binned
  4x4 (as the whole-detector plot), with the same colour scales in every row.

The trials compare the background options; add your own to `TRIALS`.  Run with the **LSST** kernel.

In [ ]:
%matplotlib inline
import os
import subprocess
import sys
import time

# where this repository is checked out (default: the folder above this notebook)
MEAS_CROWD = os.environ.get("MEAS_CROWD_DIR") or os.path.abspath("..")
# MEAS_CROWD = "/sdf/data/rubin/user/abudlong/code/build/meas_crowd_v2"
PKG = os.path.join(MEAS_CROWD, "python")
if not os.path.isdir(os.path.join(PKG, "lsst", "meas", "crowd")):
    raise FileNotFoundError(f"no meas_crowd package under {PKG}: set MEAS_CROWD above")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import lsst  # noqa: E402
import lsst.meas  # noqa: E402
for mod, sub in ((lsst, "lsst"), (lsst.meas, os.path.join("lsst", "meas"))):
    if os.path.join(PKG, sub) not in list(mod.__path__):   # lsst / lsst.meas were imported before
        mod.__path__.append(os.path.join(PKG, sub))

In [ ]:
import glob

CHECK = """
import importlib.util, sys
spec = importlib.util.spec_from_file_location("_crowdLib", sys.argv[1])
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
print(hasattr(m, "solveSparseLeastSquares"))
"""

def solverBuilt():
    """True if the compiled extension has the sparse solver (checked in a fresh process)."""
    for so in glob.glob(os.path.join(PKG, "lsst", "meas", "crowd", "_crowdLib*.so")):
        out = subprocess.run([sys.executable, "-c", CHECK, so], capture_output=True, text=True)
        if out.stdout.strip() == "True":
            return True
    return False

if solverBuilt():
    print("C++ extension with the sparse solver: OK")
else:
    print("building the C++ extension ...")
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=MEAS_CROWD,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode == 0 and solverBuilt():
        print("\nBUILT -- now restart the kernel (Kernel > Restart) and run from the top.")
    else:
        raise RuntimeError("build failed: see the output above")

In [ ]:
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from scipy.spatial import cKDTree  # noqa: E402

import lsst.meas.crowd as crowd  # noqa: E402

print("meas_crowd from:", os.path.dirname(crowd.__file__))
print("C++ kernels:", crowd.HAVE_CXX, "| sparse Eigen solver:", crowd.HAVE_SPARSE_SOLVER)
assert crowd.HAVE_SPARSE_SOLVER, "the sparse solver is not built: run the build cell, restart"

# The notebook server's CPU limit is a quota (e.g. 4 CPUs) on a node whose cores are all visible
# (e.g. 128): without a cap, NumPy's BLAS starts a thread per visible core and the kernel is
# throttled.  Cap the threads at the usable CPUs.
crowd.limitBlasThreads()

## Helpers: the numbers and the figure of each trial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm


SHOW_PADDING = False   # False: figures and numbers cover only the interior each cutout keeps
#                        (as the detector run); True: the whole fitted image, interior outlined


def attachInterior(run, name):
    """Record where the cutout's interior lies in the fitted image (uses bb, CUTOUTS, SIZE)."""
    box = run.outerHistory[-1]["exposure"].getBBox()
    run.interior = (bb.getMinX() + CUTOUTS[name][0] - box.getMinX(),
                    bb.getMinY() + CUTOUTS[name][1] - box.getMinY(), SIZE)
    return run


def _view(run):
    """(x0, y0, nx, ny) of the shown part of the fitted image: the interior, or all of it."""
    shape = run.outerHistory[-1]["exposure"].image.array.shape
    if SHOW_PADDING or getattr(run, "interior", None) is None:
        return 0, 0, shape[1], shape[0]
    x0, y0, n = run.interior
    return x0, y0, n, n


def _arrays(run):
    """Data, model, variance, bad mask and the subtracted sky of a run's final pass (shown part)."""
    h = run.outerHistory[-1]
    exp, model = h["exposure"], np.asarray(h["result"]["modelFull"], float)
    data = np.asarray(exp.image.array, float)
    var = np.asarray(exp.variance.array, float)
    bad = ((exp.mask.array & run.badBitmask) != 0) | ~np.isfinite(data) | ~(var > 0)
    raw = run.rawCrop
    raw = raw.image.array if hasattr(raw, "image") else raw[exp.getBBox()].image.array
    x0, y0, nx, ny = _view(run)
    sl = (slice(y0, y0 + ny), slice(x0, x0 + nx))
    return data[sl], model[sl], var[sl], bad[sl], (np.asarray(raw, float) - data)[sl]


def _sourcesShown(run):
    """Fitted source positions inside the shown part, in its pixel coordinates."""
    h = run.outerHistory[-1]
    box = h["exposure"].getBBox()
    xy = np.asarray(h["result"]["peaks"], float).reshape(-1, 2) - [box.getMinX(), box.getMinY()]
    x0, y0, nx, ny = _view(run)
    xy = xy - [x0, y0]
    keep = (xy[:, 0] >= -0.5) & (xy[:, 0] < nx - 0.5) & (xy[:, 1] >= -0.5) & (xy[:, 1] < ny - 0.5)
    return xy[keep]


def trialStats(run):
    """Numbers for one trial: sky subtracted, residual/sigma between and on stars, Gaia offset."""
    data, model, var, bad, sky = _arrays(run)
    z = (data - model) / np.sqrt(np.where(bad, np.inf, var))
    s = model / np.sqrt(np.where(bad, np.inf, var))
    ok = ~bad
    out = dict(sky=float(np.median(sky[ok])),
               between=float(np.median(z[ok & (s < 0.5)])),
               onStars=float(np.median(z[ok & (s > 4)])) if (ok & (s > 4)).any() else np.nan,
               robustSigma=float(1.4826 * np.median(np.abs(z[ok] - np.median(z[ok])))),
               nSources=len(_sourcesShown(run)))
    ph = getattr(run, "photometry", None)
    if ph is not None and "gaiaDmag" in ph.colnames:
        g = np.asarray(ph["gaiaMatch"], bool) & (np.asarray(ph["psfSnr"]) >= 50) \
            & np.isfinite(np.asarray(ph["gaiaDmag"], float))
        if not SHOW_PADDING and "inInterior" in ph.colnames:
            g &= np.asarray(ph["inInterior"], bool)
        d = np.asarray(ph["gaiaDmag"], float)[g]
        if len(d):
            out["gaiaDmag"] = float(np.median(d))
            out["gaiaScatter"] = float(1.4826 * np.median(np.abs(d - np.median(d))))
    return out


def binnedSigma(z, b=4):
    """Residual/sigma binned b x b as sum / sqrt(n) over the finite pixels (as plotDetector)."""
    ny, nx = (z.shape[0] // b) * b, (z.shape[1] // b) * b
    zz = z[:ny, :nx].reshape(ny // b, b, nx // b, b)
    fin = np.isfinite(zz)
    n = fin.sum(axis=(1, 3))
    tot = np.where(fin, zz, 0.0).sum(axis=(1, 3))
    return np.where(n > 0, tot / np.sqrt(np.maximum(n, 1)), np.nan)


def showTrials(name, runs, labels):
    """data | model | residual | residual/sigma | binned residual/sigma, one row per trial."""
    arrs = [_arrays(r) for r in runs]
    d0, m0, v0, b0, _ = arrs[0]
    good = ~b0
    lo, hi = np.percentile(d0[good], [1, 99.7])
    rs = 1.4826 * np.median(np.abs((d0 - m0)[good] - np.median((d0 - m0)[good])))
    dnorm = AsinhNorm(linear_width=max(3 * rs, 1e-9), vmin=lo, vmax=hi)
    fig, axes = plt.subplots(len(runs), 5, figsize=(23, 4.4 * len(runs)), squeeze=False)
    for row, ((data, model, var, bad, sky), run, lab) in enumerate(zip(arrs, runs, labels)):
        st = trialStats(run)
        resid = np.where(bad, np.nan, data - model)
        z = resid / np.sqrt(np.where(bad, np.nan, var))
        panels = [(np.where(bad, np.nan, data), dict(cmap="viridis", norm=dnorm), "data (sky-subtracted)"),
                  (np.where(bad, np.nan, model), dict(cmap="viridis", norm=dnorm), "model"),
                  (resid, dict(cmap="RdBu_r", vmin=-4 * rs, vmax=4 * rs), "residual"),
                  (z, dict(cmap="RdBu_r", vmin=-4, vmax=4), "residual / sigma"),
                  (binnedSigma(z), dict(cmap="RdBu_r", vmin=-4, vmax=4),
                   "residual / sigma, binned 4x4 (sum/sqrt n)")]
        for ax, (img, kw, title) in zip(axes[row], panels):
            ax.imshow(img, origin="lower", interpolation="nearest", **kw)
            ax.set_xticks([]), ax.set_yticks([])
            ax.set_title(title, fontsize=10)
            ax.set_facecolor("magenta")
        axes[row, 0].set_ylabel(lab, fontsize=11)
        axes[row, 3].set_title(f"residual / sigma: between stars {st['between']:+.3f}, "
                               f"on stars {st['onStars']:+.3f}", fontsize=9)
        axes[row, 2].set_title(f"residual (sky {st['sky']:+.2f} counts, robust sigma "
                               f"{st['robustSigma']:.3f}/px)", fontsize=9)
        axes[row, 1].set_title(f"model ({st['nSources']} sources)", fontsize=10)
    fig.suptitle(f"{name}: rows = trials; magenta = bad / unfitted pixels; "
                 f"{'whole fitted image' if SHOW_PADDING else 'interior only (padding not shown)'}",
                 fontsize=12)
    fig.tight_layout()
    return fig

## Settings, visit image, cutouts

In [ ]:
import contextlib, io, time  # noqa: E401

VISIT, DETECTOR = 2025062100433, 144
REPO, COLLECTION = "dp2_prep", "LSSTCam/runs/DRP/DP2"
SIZE, PAD = 372, 30
OUT = "sky_trials"
os.makedirs(OUT, exist_ok=True)

TRIALS = {                                   # label -> options on top of FAST_STELLAR_FIELD_CONFIG
    "fast default (PIFF constant, 40 stars)": dict(),
    "light first pass, no clean-up": dict(lightFirstPass=True, lightDropNearBrightFwhm=0),
    "light first pass + clean-up": dict(lightFirstPass=True),
}

from lsst.daf.butler import Butler  # noqa: E402
butler = Butler(REPO, collections=COLLECTION)
refs = butler.query_datasets("visit_image", collections="LSSTCam/runs/*", find_first=False,
                             where=f"instrument='LSSTCam' AND visit={VISIT} AND detector={DETECTOR}")
visitImage = butler.get(refs[0])
bb = visitImage.getBBox()
W, H = bb.getWidth(), bb.getHeight()
CUTOUTS = {                                  # name -> lower-left corner of the cutout
    "bottom-left": (0, 0), "bottom-right": (W - SIZE, 0),
    "top-left": (0, H - SIZE), "top-right": (W - SIZE, H - SIZE),
    "centre": ((W - SIZE) // 2, (H - SIZE) // 2),
}
specs = [dict(x0=bb.getMinX() + x, y0=bb.getMinY() + y, w=SIZE, h=SIZE, name=n)
         for n, (x, y) in CUTOUTS.items()]
sess = crowd.prepareSession(VISIT, DETECTOR, specs, pad=PAD, psfSource="visit", visitImage=visitImage)
crowd.attachGaia(sess, visitImage=visitImage, repo=REPO)
print({n: f"x {x}-{x + SIZE - 1}, y {y}-{y + SIZE - 1}" for n, (x, y) in CUTOUTS.items()})

## Run every trial on every cutout

Each fit's log goes to `sky_trials/logs/<cutout>__<trial>.log` as it runs; if the kernel dies, the last
log and `sky_trials/logs/crash.txt` show where.

In [ ]:
import faulthandler, gc, resource, sys, types  # noqa: E401

LOGDIR = os.path.join(OUT, "logs")      # one log per trial, written as it runs: after a dead
os.makedirs(LOGDIR, exist_ok=True)      #   kernel, the last one shows where it stopped
CRASH = open(os.path.join(LOGDIR, "crash.txt"), "w")
faulthandler.enable(CRASH, all_threads=True)   # a hard crash (segfault) writes its traceback here


def memGB():
    """Peak memory of this kernel so far [GB]."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6


def slim(run):
    """Keep only what the table and figures need (the final pass), so 20 runs fit in memory."""
    h = run.outerHistory[-1]
    return types.SimpleNamespace(
        outerHistory=[dict(exposure=h["exposure"],
                           result=dict(modelFull=h["result"]["modelFull"], peaks=h["result"]["peaks"]))],
        rawCrop=run.rawCrop, badBitmask=run.badBitmask, photometry=getattr(run, "photometry", None))


runs, rows = {}, []
for name in CUTOUTS:
    runs[name] = {}
    for label, opts in TRIALS.items():
        logPath = os.path.join(LOGDIR, f"{name}__{label}".replace(" ", "_").replace("/", "-") + ".log")
        print(f"{name} / {label} ... (peak memory so far {memGB():.1f} GB, log {logPath})", flush=True)
        cfg = crowd.makeConfig(crowd.FAST_STELLAR_FIELD_CONFIG, **opts)
        t0 = time.perf_counter()
        with open(logPath, "w", buffering=1) as logf, contextlib.redirect_stdout(logf):
            try:
                r = crowd.runSessionGaia(sess, cfg, names=[name], plotFinal=False)
            except Exception as e:
                print(f"EXCEPTION {type(e).__name__}: {e}")
                r = {}
        dt = time.perf_counter() - t0
        if name not in r:
            tail = open(logPath).read().splitlines()[-25:]
            print(f"  FAILED -- end of its log:\n    " + "\n    ".join(tail))
            continue
        runs[name][label] = attachInterior(slim(r[name]), name)
        rows.append((name, label, dt, trialStats(runs[name][label])))
        del r
        gc.collect()
        print(f"  done in {dt:.1f} s", flush=True)

hdr = (f"{'cutout':13s} {'trial':24s} {'time':>5s} {'sky':>7s} {'between':>8s} {'on stars':>8s} "
       f"{'scatter':>7s} {'sources':>7s} {'psfMag-G':>8s} {'(scatter)':>9s}")
print(hdr + "\n" + "-" * len(hdr))
for name, label, dt, st in rows:
    print(f"{name:13s} {label:24s} {dt:4.1f}s {st['sky']:+7.2f} {st['between']:+8.3f} "
          f"{st['onStars']:+8.3f} {st['robustSigma']:7.3f} {st['nSources']:7d} "
          f"{st.get('gaiaDmag', float('nan')):+8.3f} {st.get('gaiaScatter', float('nan')):9.3f}")
print("\nsky: counts subtracted (median); between / on stars: median residual/sigma where the model "
      "is < 0.5 / > 4 sigma (0 is ideal); scatter: robust residual/sigma per pixel (1 is ideal)")

## Data, model and residuals of every trial

One figure per cutout, one row per trial.  Data and model share a colour scale (asinh); residual
panels use the same range in every row, so differences between trials show directly.  Figures are
also saved to `sky_trials/`.

In [ ]:
for name in CUTOUTS:
    if not runs.get(name):
        continue
    labels = list(runs[name])
    fig = showTrials(f"{name} (x0, y0 = {CUTOUTS[name]})", [runs[name][k] for k in labels], labels)
    fig.savefig(os.path.join(OUT, f"{name}.png"), dpi=80)
    plt.show()

## The final pass of one trial, every cutout

One row per cutout, each on its own colour scale; set `FINAL` to another trial to compare.

In [ ]:
def showFinal(runs, trial, names=None, sources=True):
    """One row per cutout: the final pass of one trial (data | model | residual | residual/sigma |
    binned), each row on its own colour scale; dots on the data = fitted sources."""
    names = [n for n in (names or runs) if trial in runs[n]]
    fig, axes = plt.subplots(len(names), 5, figsize=(23, 4.6 * len(names)), squeeze=False)
    for row, name in enumerate(names):
        run = runs[name][trial]
        data, model, var, bad, sky = _arrays(run)
        st = trialStats(run)
        good = ~bad
        lo, hi = np.percentile(data[good], [1, 99.7])
        resid = np.where(bad, np.nan, data - model)
        rs = st["robustSigma"] * float(np.sqrt(np.median(var[good])))
        dnorm = AsinhNorm(linear_width=max(3 * rs, 1e-9), vmin=lo, vmax=hi)
        z = resid / np.sqrt(np.where(bad, np.nan, var))
        panels = [(np.where(bad, np.nan, data), dict(cmap="viridis", norm=dnorm),
                   "data (sky-subtracted)"),
                  (np.where(bad, np.nan, model), dict(cmap="viridis", norm=dnorm),
                   f"model ({st['nSources']} sources)"),
                  (resid, dict(cmap="RdBu_r", vmin=-4 * rs, vmax=4 * rs),
                   f"residual (sky {st['sky']:+.1f} counts)"),
                  (z, dict(cmap="RdBu_r", vmin=-4, vmax=4),
                   f"resid/sigma: between {st['between']:+.3f}, on stars {st['onStars']:+.3f}, "
                   f"scatter {st['robustSigma']:.3f}"),
                  (binnedSigma(z), dict(cmap="RdBu_r", vmin=-4, vmax=4),
                   "residual / sigma, binned 4x4")]
        for ax, (img, kw, title) in zip(axes[row], panels):
            ax.imshow(img, origin="lower", interpolation="nearest", **kw)
            ax.set_xticks([]), ax.set_yticks([])
            ax.set_facecolor("magenta")
            ax.set_title(title, fontsize=9.5)
        if sources:
            xy = _sourcesShown(run)
            axes[row, 0].plot(xy[:, 0], xy[:, 1], ".", color="red", ms=1.5)
            axes[row, 0].set_xlim(-0.5, data.shape[1] - 0.5)
            axes[row, 0].set_ylim(-0.5, data.shape[0] - 0.5)
        if SHOW_PADDING and getattr(run, "interior", None) is not None:
            ix0, iy0, n = run.interior
            for ax in axes[row]:
                ax.add_patch(plt.Rectangle((ix0 - 0.5, iy0 - 0.5), n, n, fill=False,
                                           edgecolor="orange", lw=2, ls="--"))
        axes[row, 0].set_ylabel(f"{name}\n{CUTOUTS[name]}", fontsize=11)
    fig.suptitle(f"final pass: {trial}  (red dots: fitted sources; "
                 + ("orange box: the interior kept, outside it the padding)" if SHOW_PADDING
                    else "interior only: the 30 px padding is not shown)"),
                 fontsize=13)
    fig.tight_layout()
    return fig


FINAL = "light first pass + clean-up"       # the trial to show
fig = showFinal(runs, FINAL)
fig.savefig(os.path.join(OUT, "final_pass.png"), dpi=110)
plt.show()

## Noise or PSF? A check of the final pass

chi2 per pixel in each star's core against its S/N (about 1 everywhere for noise; rising with S/N for a PSF error), and the stacked residual / flux of the bright stars with its radial profile.

In [ ]:
# PSF check: is the leftover structure noise or a PSF error?  (paste after the trials ran)
# 1. chi2 per pixel in each star's core vs its S/N: ~1 at every S/N for noise; rising with S/N
#    means the model's shape is off (the mismatch grows with flux, the noise only with sqrt(flux)).
# 2. stacked residual / flux of the bright stars: noise averages away; a PSF error leaves a
#    pattern (e.g. a dark core and a bright ring = model too narrow), shown with its radial profile.
CHECK = FINAL                      # the trial to check
SNR_BINS = [10, 30, 100, 300, 1000, 1e9]
HALF, CORE = 12, 3                 # stack half-size; core radius for chi2 [px]


def psfCheck(run, name):
    data, model, var, bad, _ = _arrays(run)
    ph = run.photometry
    box = run.outerHistory[-1]["exposure"].getBBox()
    x0, y0, nx, ny = _view(run)
    x = np.asarray(ph["x"], float) - box.getMinX() - x0
    y = np.asarray(ph["y"], float) - box.getMinY() - y0
    snr, flux = np.asarray(ph["psfSnr"], float), np.asarray(ph["psfFlux"], float)
    z = (data - model) / np.sqrt(np.where(bad, np.inf, var))
    yy, xx = np.mgrid[-HALF:HALF + 1, -HALF:HALF + 1]
    core = xx ** 2 + yy ** 2 <= CORE ** 2
    chi2, stack, w = [], [], []
    for i in range(len(x)):
        ix, iy = int(round(x[i])), int(round(y[i]))
        if not (HALF <= ix < nx - HALF and HALF <= iy < ny - HALF) or not np.isfinite(snr[i]):
            chi2.append(np.nan)
            continue
        sl = (slice(iy - HALF, iy + HALF + 1), slice(ix - HALF, ix + HALF + 1))
        zc, bc = z[sl], bad[sl]
        chi2.append(float(np.mean(zc[core & ~bc] ** 2)) if (core & ~bc).sum() > 10 else np.nan)
        if snr[i] >= 100 and not bc.any() and flux[i] > 0:
            stack.append((data - model)[sl] / flux[i])
            w.append(1.0)
    chi2 = np.asarray(chi2)
    rows = []
    for lo, hi in zip(SNR_BINS[:-1], SNR_BINS[1:]):
        m = (snr >= lo) & (snr < hi) & np.isfinite(chi2)
        if m.sum():
            rows.append((lo, hi, int(m.sum()), float(np.median(chi2[m]))))
    return rows, (np.mean(stack, axis=0) if stack else None), \
        (np.std(stack, axis=0) / np.sqrt(len(stack)) if len(stack) > 1 else None), len(stack)


names = [n for n in runs if CHECK in runs[n]]
fig, axes = plt.subplots(len(names), 2, figsize=(11, 3.8 * len(names)), squeeze=False)
print(f"chi2 per pixel within {CORE} px of each star (median per S/N bin; ~1 = noise only)")
for row, name in enumerate(names):
    rows_, mean, err, n = psfCheck(runs[name][CHECK], name)
    print(f"  {name:13s} " + "  ".join(f"S/N {lo:g}-{hi if hi < 1e8 else 'up':}: {c:.2f} ({k})"
                                       for lo, hi, k, c in rows_))
    if mean is None:
        continue
    lim = np.nanmax(np.abs(mean))
    axes[row, 0].imshow(mean, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[row, 0].set_title(f"{name}: mean residual / flux of {n} stars with S/N >= 100", fontsize=9)
    axes[row, 0].set_xticks([]), axes[row, 0].set_yticks([])
    yy, xx = np.mgrid[-HALF:HALF + 1, -HALF:HALF + 1]
    r = np.hypot(xx, yy).round().astype(int)
    prof = np.array([mean[r == k].mean() for k in range(HALF + 1)])
    perr = np.array([np.sqrt(np.sum(err[r == k] ** 2)) / (r == k).sum() for k in range(HALF + 1)])
    axes[row, 1].errorbar(np.arange(HALF + 1), prof, perr, fmt="o-", ms=3)
    axes[row, 1].axhline(0, color="k", lw=0.8)
    axes[row, 1].set_xlabel("radius [px]")
    axes[row, 1].set_ylabel("mean residual / flux per px")
    axes[row, 1].set_title("radial profile (error bars: star-to-star scatter / sqrt N)", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "psf_check.png"), dpi=110)
plt.show()
print("\nReading it: chi2 ~1 in every bin and a stack consistent with 0 -> the spots are noise "
      "(and faint stars below the detection limit).  chi2 rising with S/N and a coherent "
      "stack pattern -> the PSF model is off for bright stars.")